# Plot LaTeX table of results of Coordbench

In [ ]:
import os
import argparse
import logging
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path
from itertools import groupby
from typing import NamedTuple

import pandas as pd

In [2]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../../../.."))
DEFAULT_RESULTS_ROOT = os.path.join(REPO_ROOT, "results", "coordbench")

## For spatial split

In [3]:
# (category, header label, [(dataset, task), ...]); multiple pairs are averaged into one cell.
COLUMNS = [
    ("Terrain", "Elevation", [("satclip-elevation", "elevation")]),
    ("Climate", "Air Temp.", [("satclip-air-temp", "air_temperature")]),
    ("Climate", "Bioclim. (BT)", [("bt-bioclim", f"bioclim_{i:02d}") for i in range(1, 20)]),
    (r"Ecology \& Environment", "Ecoregions", [("satclip-ecoregion", "ecoregion_class_index")]),
    (r"Ecology \& Environment", "Biomes", [("satclip-biome", "biome_class_index")]),
    (r"Ecology \& Environment", "SoilGrids", [("soilgrids-phh2o", "phh2o"), ("soilgrids-soc", "soc")]),
    (r"Socioeconomic \& Agriculture", "Wealth (SB)", [("sustainbench-asset", "asset_index")]),
    (r"Socioeconomic \& Agriculture", "Population (BT)", [("bt-population", "pop_density")]),
    (r"Socioeconomic \& Agriculture", "Crop Harvest (BT)", [("bt-cropharvest", "crop_type")]),
]

In [4]:
# Row blocks separated by \midrule; the last block is highlighted and bolded.
MODEL_GROUPS = [
    [("sinr", "SINR"), ("csp-inat", "CSP-iNat"), ("csp-fmow", "CSP-fMoW"), ("geoclip", "GeoCLIP"),
     ("satclip", "SatCLIP"), ("gair", "GAIR"), ("tte", "TTE"), ("taxabind", "Taxabind"),
     ("sled", "SLED"), ("mind", "MIND")],
    [("gtloc", "GT-Loc"), ("climplicit", "Climplicit")],
    [("t_satclip_doy_1M", "T-SatCLIP-DoY-1M")],
]

In [5]:
TEMPORAL_MODELS = {"gtloc", "climplicit", "t_satclip_doy_1M"}
TEMPORAL_MODEL_SUFFIX = {
    "climplicit": "_year_monthly",
    "gtloc": "_year_monthly",
    "t_satclip_doy_1M": "_year_daily",
}
TEMPORAL_AGG = "None/mean"


In [ ]:
def resolve_spatial(model, dataset):
    # Temporal encoders are evaluated on synthetically timestamped variants of the atemporal datasets.
    suffix = TEMPORAL_MODEL_SUFFIX.get(model)
    return (dataset, "") if suffix is None else (dataset + suffix, TEMPORAL_AGG)


def lookup(df, model, dataset, task, resolve=resolve_spatial):
    dataset, agg = resolve(model, dataset)
    rows = df[(df.model_name == model) & (df.dataset == dataset) & (df.temporal_embedding_aggregation == agg)]
    return mean_std(rows[rows.task == task])

In [7]:
def mean_std(rows):
    # ci_lower/ci_upper are metric -/+ std over folds.
    return (rows.metric_value.mean(), ((rows.ci_upper - rows.ci_lower) / 2).mean()) if len(rows) else None


In [ ]:
def cell_value(df, model, pairs, resolve=resolve_spatial):
    values = [v for d, t in pairs if (v := lookup(df, model, d, t, resolve)) is not None]
    if not values:
        return None
    means, stds = zip(*values)
    return sum(means) / len(means), sum(stds) / len(stds)


def format_cell(value, bold=False):
    if value is None:
        return "--"
    mean, std = value
    body = rf"\mathbf{{{mean:.2f} \pm {std:.2f}}}" if bold else rf"{mean:.2f} \pm {std:.2f}"
    return f"${body}$"

In [ ]:
class Column(NamedTuple):
    category: str
    label: str
    pairs: list  # [(dataset, task), ...]; averaged into one cell
    resolve: Callable = resolve_spatial  # (model, dataset) -> (dataset name, temporal aggregation)
    source: str | None = None  # result-file stem; separates datasets that share a name


def load_results(results_root, split, directories=None):
    paths = [p for p in Path(results_root).rglob("*.csv") if directories is None or p.parent.name in directories]
    df = pd.concat([pd.read_csv(p).assign(source=p.stem) for p in paths], ignore_index=True)
    df["temporal_embedding_aggregation"] = df.temporal_embedding_aggregation.fillna("")
    # Linear probe for regression, KNN-5 for classification.
    return df[(df.split == split) & (df.method == df.task_type.map({"regression": "linear", "classification": "knn5"}))]


def render_table(df, columns, name, caption, note):
    frames = [df if c.source is None else df[df.source == c.source] for c in columns]
    # Best per column is the highest mean; ties are compared at the printed precision.
    values = {
        m: [cell_value(f, m, c.pairs, c.resolve) for c, f in zip(columns, frames)]
        for g in MODEL_GROUPS for m, _ in g
    }
    best = [
        max((round(v[j][0], 2) for v in values.values() if v[j] is not None), default=None)
        for j in range(len(columns))
    ]

    groups = [(cat, len(list(cols))) for cat, cols in groupby(columns, key=lambda c: c.category)]
    starts = [2 + sum(span for _, span in groups[:i]) for i in range(len(groups))]

    out = [
        f"% {'=' * 20} {name[0].upper() + name[1:]} {'=' * 20}",
        r"\begin{table*}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{tab:{name.lower().replace(' ', '_')}}}",
        r"\renewcommand{\arraystretch}{1.15}",
        r"\setlength{\tabcolsep}{5pt}",
        "",
        r"\resizebox{\textwidth}{!}{%",
        rf"\begin{{tabular}}{{l{'c' * len(columns)}}}",
        r"\toprule",
        "",
        "& " + "\n& ".join(rf"\multicolumn{{{n}}}{{c}}{{\textbf{{{cat}}}}}" for cat, n in groups) + r" \\",
        "",
        *(rf"\cmidrule(lr){{{s}-{s + n - 1}}}" for s, (_, n) in zip(starts, groups)),
        "",
        r"\textbf{Model}" + "".join(f"\n& \\textbf{{{c.label}}}" for c in columns) + r" \\",
        "",
        r"\midrule",
    ]
    for i, group in enumerate(MODEL_GROUPS):
        last = i == len(MODEL_GROUPS) - 1
        out.append("")
        for model, label in group:
            cells = " & ".join(
                format_cell(v, bold=v is not None and round(v[0], 2) == best[j])
                for j, v in enumerate(values[model])
            )
            if last:
                out += [r"\rowcolor{gray!15}", rf"\textbf{{{label}}} & {cells} \\"]
            else:
                out.append(rf"{label} & {cells} \\")
        out += ["", r"\bottomrule" if last else r"\midrule"]
    out += [
        r"\end{tabular}%",
        "}",
        "",
        r"\vspace{0.1cm}",
        r"\begin{minipage}{\textwidth}",
        r"\footnotesize",
        rf"\textit{{Note:}} {note}",
        r"\end{minipage}",
        "",
        r"\end{table*}",
    ]
    return "\n".join(out)


def make_table(results_root, split="spatial", cell_deg=None, temporal_grid_size=None):
    df = load_results(results_root, split)
    if cell_deg is not None:
        df = df[df.cell_deg == cell_deg]
    if temporal_grid_size is not None:
        df = df[df.temporal_grid_size == temporal_grid_size]
    name = split + (" ID" if split in ("random", "spatial") else "")
    return render_table(
        df,
        [Column(*c) for c in COLUMNS],
        name,
        f"Model comparison under {name} evaluation.",
        "BT = Better Together; SB = SustainBench. Best result per column in bold.",
    )

In [ ]:
print(make_table(DEFAULT_RESULTS_ROOT, split="random"))

In [ ]:
print(make_table(DEFAULT_RESULTS_ROOT, split="spatial"))

## For temporal datasets

In [ ]:
# Sweeps that cover the timestamped datasets. Overlapping rows across them agree, so pooling is safe.
TEMPORAL_DIRS = {"resolution_matched_temporal_aggregation", "electrical_load", "temporal_split"}

# Tasks present for every encoder, so each cell averages the same set across models.
CHELSA_DAILY_TASKS = ["hurs", "pr", "ps", "rsds", "sfcWind", "tas", "tasmax", "tasmin", "tz", "we"]
CHELSA_MONTHLY_TASKS = ["cmi_log", "hurs", "pet_log", "pr", "rsds", "tas", "tasmax", "tasmin"]
ERA5_TASKS = ["d2m", "sde", "skt", "snowc", "sp", "ssrd", "stl1", "stl2", "stl3", "stl4", "strd",
              "swvl1", "swvl2", "swvl3", "swvl4", "t2m", "tp", "u10", "v10"]
ELECTRIC_LOAD_PAIRS = [
    (f"usa_daily_electric_load_{year}-{spatial_agg}", task)
    for year in range(2016, 2024)
    for spatial_agg in ("statistical", "median_and_iqr")
    for task in ("max", "max_log", "median", "median_log", "min", "min_log")
]


def variant(default=("{d}", ""), **by_model):
    """Build a resolver from (dataset-name template, aggregation) pairs; ``{d}`` is the base dataset."""
    def resolve(model, dataset):
        template, agg = by_model.get(model, default)
        return template.format(d=dataset), agg
    return resolve


# A temporal encoder pools embeddings over a window coarser than its native resolution
# (Climplicit and GT-Loc are monthly, T-SatCLIP is daily); otherwise it sees each row's timestamp.
YEARLY = "TemporalResolution.YEARLY/mean"
POOL_MONTHLY = ("{d}_monthly_discretization", YEARLY)
POOL_DAILY = ("{d}_daily_discretization", YEARLY)
NATIVE = variant()

TEMPORAL_COLUMNS = [
    Column("Climate", "CHELSA-daily", [("chelsa", t) for t in CHELSA_DAILY_TASKS], NATIVE, "chelsa_daily"),
    Column("Climate", "CHELSA-monthly", [("chelsa", t) for t in CHELSA_MONTHLY_TASKS],
           variant(t_satclip_doy_1M=("chelsa_daily_discretization", "TemporalResolution.MONTHLY/mean")),
           "chelsa_monthly"),
    Column("Climate", "ERA5", [("era5_ecmwf", t) for t in ERA5_TASKS], NATIVE, "era5_ecmwf"),
    Column("Ecological", "US Trees", [("dm-us_trees", "us_trees")], NATIVE, "dm-us_trees"),
    Column(r"Socioeconomic \& Agriculture", "Global Population",
           [("global_population_global", "log_population")],
           variant(climplicit=POOL_MONTHLY, gtloc=POOL_MONTHLY, t_satclip_doy_1M=POOL_DAILY),
           "global_population"),
    Column(r"Socioeconomic \& Agriculture", "Nightlights", [("viirs_night_lights", "night_lights")],
           variant(climplicit=POOL_MONTHLY, gtloc=POOL_MONTHLY), "viirs_night_lights"),
    Column(r"Socioeconomic \& Agriculture", "USA Electrical Load", ELECTRIC_LOAD_PAIRS,
           variant(default=("{d}", "None/mean")), "usa_electric_load_daily"),
]

# split -> (table name, evaluation phrase used in the caption)
TEMPORAL_TABLES = {
    "random": ("temporal ID", "temporal in-distribution (ID)"),
    "temporal": ("temporal OOD", "temporal out-of-distribution (OOD)"),
}


def make_temporal_table(results_root, split="random", temporal_grid_size=None):
    if split == "temporal" and temporal_grid_size is None:
        raise ValueError("The temporal split is swept over grid sizes; pass temporal_grid_size in seconds.")
    df = load_results(results_root, split, TEMPORAL_DIRS)
    if temporal_grid_size is not None:
        df = df[df.temporal_grid_size == temporal_grid_size]
    name, phrase = TEMPORAL_TABLES[split]
    return render_table(
        df, TEMPORAL_COLUMNS, name, f"Model comparison under {phrase} evaluation.",
        "Best result per column in bold.",
    )

In [ ]:
print(make_temporal_table(DEFAULT_RESULTS_ROOT, split="random"))

In [ ]:
TEMPORAL_GRID_SIZE = 2_419_200  # seconds; 4-week holdout window (also available: 604_800, 7_862_400)
print(make_temporal_table(DEFAULT_RESULTS_ROOT, split="temporal", temporal_grid_size=TEMPORAL_GRID_SIZE))